### Zoteroize and Obsidianize a Perplexity Dialogue

In a Perplexity dialogue saved by Perplexity itself, replace the citation numbers with matching Obsidian literature note or Zotero item links

**It's half done.**  If you want to process plain perplexity outputs like this code does, but well, it might make sense to hack the code for relinking `Save my Chatbot` exports.

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
output_file_perplex = tmp_dir / "tmp_new_cites_perplexity_example.md"  # processed raw perplexity output

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


In [3]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [4]:
# Make a lookup dict: zotero DB item URL to bibtex citekey
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [5]:
# Collect info about each zotero DB item that has a URL


lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}

zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if not (title := pdat.get('title')):
        continue

    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], title=title, hasLitNote=citekeyThis in lit_note_file_stems))

    if url := pdat.get('url'):
        if normalized_url :=rfw.normalize_url(url):
            citekeysForURL[normalized_url].append(citekeyThis)

if repeatedURLs := {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}:
    print(f"Found {len(repeatedURLs)} URLs with > 1 parent (citekey)")
    for url, citekeys in repeatedURLs.items():
        print(f"{', '.join(citekeys)}\n\t{url}")
    raise Exception(f'Not built for repeated URLS')

citekey_to_url = {citekeys[0]: url for url, citekeys in citekeysForURL.items()}
url_to_citekey = {url: citekey for citekey, url in citekey_to_url.items()}

zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

if sum(hasNoURL := zot_db_items.url.isna()):
    print(f"Dropping {sum(hasNoURL)} of {len(zot_db_items)} zotero entries with no URL:")
    display((zot_db_items_no_url := zot_db_items[hasNoURL]).head())
    zot_db_items = zot_db_items[~hasNoURL]

Dropping 149 of 1681 zotero entries with no URL:


,citekey,zotkey,title,hasLitNote,url
1,MMSDataModelSummary_v5.2,8XJHRYMU,MMS Data Model Package Summary v5.2,False,NaN
16,Seals99irradFrcstDiag,WYP9J7EU,The heart of suny irradiance forecasting,False,NaN
92,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,Testing increases suggestibility for narrative...,False,NaN
153,Gaur20attribModellingRvw,HLHKVCLX,Attribution modelling in marketing: Literature...,False,NaN
200,Wang19predOptcoolLdFrcst,R7TJLE7Y,Cooling load forecasting-based predictive opti...,False,NaN


In [6]:
#zot_db_items.url.isna()
zot_db_items

,citekey,zotkey,title,hasLitNote,url
0,Zbili21entMutInfoQuickEasyEst,VVUZBQM2,A quick and easy way to estimate entropy and m...,True,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197
2,Gerber09normsMotiveVote,6V73VWME,Descriptive social norms and motivation to vot...,False,https://www.journals.uchicago.edu/doi/abs/10.1...
3,Poston13politAdsVizAuralMeaning,6VPD3STC,Political advertising in the 2012 presidential...,False,https://scholarworks.boisestate.edu/td/602
4,Deaton21altruisByCountry,YIN5FL7M,An exploration of global altruistic variations...,False,https://rave.ohiolink.edu/etdc/view?acc_num=xu...
5,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,Topic analysis in news via sparse learning: a ...,False,https://www.sciencedirect.com/science/article/...
...,...,...,...,...,...
1670,Oracle19evDetDisaggAMI,JI47UFCR,AMI-based EV Detection & Disaggregation,False,https://www.oracle.com/a/ocom/docs/industries/...
1672,Bidgely19amInsightsRprt,EKLZCYP5,AMI-Driven Insights Report,False,https://www.idcutilitiessummit.com/index/resou...
1674,Hare18disaggHmLdDmdResp,WA8IQAXP,Disaggregation of residential home energy via ...,False,https://dspace.mit.edu/handle/1721.1/117983
1675,Rehman21LoadDisaggThesis,8MAAZN8P,Load Disaggregation: Towards Energy Efficient ...,False,https://openrepository.aut.ac.nz/handle/10292/...


In [7]:
zot_db_items

,citekey,zotkey,title,hasLitNote,url
0,Zbili21entMutInfoQuickEasyEst,VVUZBQM2,A quick and easy way to estimate entropy and m...,True,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197
2,Gerber09normsMotiveVote,6V73VWME,Descriptive social norms and motivation to vot...,False,https://www.journals.uchicago.edu/doi/abs/10.1...
3,Poston13politAdsVizAuralMeaning,6VPD3STC,Political advertising in the 2012 presidential...,False,https://scholarworks.boisestate.edu/td/602
4,Deaton21altruisByCountry,YIN5FL7M,An exploration of global altruistic variations...,False,https://rave.ohiolink.edu/etdc/view?acc_num=xu...
5,Calafiore17newsTopicSparseLrnPresid,KBL5IGVD,Topic analysis in news via sparse learning: a ...,False,https://www.sciencedirect.com/science/article/...
...,...,...,...,...,...
1670,Oracle19evDetDisaggAMI,JI47UFCR,AMI-based EV Detection & Disaggregation,False,https://www.oracle.com/a/ocom/docs/industries/...
1672,Bidgely19amInsightsRprt,EKLZCYP5,AMI-Driven Insights Report,False,https://www.idcutilitiessummit.com/index/resou...
1674,Hare18disaggHmLdDmdResp,WA8IQAXP,Disaggregation of residential home energy via ...,False,https://dspace.mit.edu/handle/1721.1/117983
1675,Rehman21LoadDisaggThesis,8MAAZN8P,Load Disaggregation: Towards Energy Efficient ...,False,https://openrepository.aut.ac.nz/handle/10292/...


## For raw perplexity dialog markdown

#### R1-inspired version

This partly works, but it's only looking up stuff by URL. It's not getting the body link titles, it doesn't modify the citation table links, and it doesn't look up zotero items by title, as a fallback when the right entry in the zot db is missing a URL or as a different one for a paper with the same title

In [8]:
from pathlib import Path
import re
from urllib.parse import urlparse, urlunparse
from collections import defaultdict
import pandas as pd

def zotero_item_link(zotero_item_key: str, link_text: str) -> str:
    """Create a Markdown link to a Zotero item using its library key.
    
    Args:
        zotero_item_key: Zotero item key from library
        link_text: Display text for the link
        
    Returns:
        Markdown link string in format [text](zotero://...)
    """
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def normalize_url(url: str) -> str:
    """Standardize URL format for consistent comparisons.
    
    Converts to lowercase and strips trailing slashes from path.
    
    Args:
        url: Any URL string
        
    Returns:
        Normalized URL string
    """
    parsed = urlparse(url.lower())
    cleaned_path = parsed.path.rstrip('/')
    return urlunparse(parsed._replace(path=cleaned_path))

def relink_perplexity_export(
    perplexity_doc: Path,
    zot_db_items: pd.DataFrame,
    output_file: Path
) -> None:
    """Replace numeric citations with Obsidian/Zotero links while preserving original structure.
    
    Processing logic:
    1. Normalizes all URLs for comparison
    2. Creates mapping from normalized URLs to Zotero item metadata
    3. Replaces body citations with appropriate links
    4. Updates citations section with matching references
    
    Args:
        perplexity_doc: Path to input Markdown file
        zot_db_items: DataFrame containing:
            - citekey: Obsidian note ID
            - zotkey: Zotero item key
            - hasLitNote: Boolean for existing literature note 
            - url: Item URL
        output_file: Path for output file
        
    Raises:
        ValueError: If input document structure is invalid
    """
    # Validate input dataframe structure
    required_columns = {'citekey', 'zotkey', 'hasLitNote', 'url'}
    if not required_columns.issubset(zot_db_items.columns):
        missing = required_columns - set(zot_db_items.columns)
        raise ValueError(f"Missing required columns in zot_db_items: {', '.join(missing)}")

    # Preprocess Zotero data
    zot_db_items = zot_db_items.copy()
    zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
    url_to_zot_info = {
        row.url: row 
        for row in zot_db_items.itertuples(index=False)
    }

    # Read and split document
    content = perplexity_doc.read_text(encoding='utf-8')
    try:
        body, citations = content.split("\nCitations:\n", 1)
    except ValueError as e:
        raise ValueError("Invalid document structure - missing citations section") from e

    # Extract citation URLs
    citation_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)
    citation_map = {
        m.group('num'): normalize_url(m.group('url')) 
        for m in citation_matches
    }

    def create_reference(doc_url: str, cite_num: str) -> str:
        """Generate appropriate reference link based on available metadata."""
        if not doc_url:
            return f'[{cite_num}]'
            
        item = url_to_zot_info.get(doc_url)
        if not item:
            return f'[{cite_num}]'

        if item.hasLitNote:
            return f'[[{item.citekey}]]'
            
        link_text = f"{item.citekey}→{item.zotkey[:6]}"
        return zotero_item_link(item.zotkey, link_text)

    # Process body content
    body_processed = re.sub(
        r'\[(\d+)\]',
        lambda m: f' {create_reference(citation_map.get(m.group(1)), m.group(1))}',
        body
    )

    # Process citations section
    def update_citation(m: re.Match) -> str:
        num = m.group('num')
        url = normalize_url(m.group('url'))
        if url in url_to_zot_info:
            # a hack to redo this here
            ref = create_reference(url, num)
            return f'[{num}] =={ref}== {m.group('url')}'
        
        return f'[{num}] {m.group('url')}'

    citations_processed = re.sub(
        r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)',
        update_citation,
        citations,
        flags=re.M
    )

    # Write output
    output_file.write_text(
        f"{body_processed}\nCitations:\n{citations_processed}", 
        encoding='utf-8'
    )

In [9]:
relink_perplexity_export(perplexity_dialog_file, zot_db_items, output_file_perplex)
ic(perplexity_dialog_file, output_file_perplex)
print('Done.')

ic| perplexity_dialog_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_example.md')
    output_file_perplex: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')


Done.


#### "My" version

In [10]:

# # Functions for replacing references in perplexity's dialog copy with links to existing obsidian notes or zotero items 

# def zotero_item_link(zotero_item_key, link_text):
#     """Makes a link to a zotero item, given its key"""
#     return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url):
#     """Convert a URL to a standard form, so that it can be string-compared to the same URL
#     written by a different program, but which is also normalized by this function."""
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

# def replace_perplexity_dialogue_links(perplexity_doc, zot_db_items, output_file):
#     """Replace numeric citations in a perplexity dialog document with links to matching 
#     obsidian literature notes or to zotero items.  A 'match' is determined when the URL 
#     in the perplexity doc matches a zotero item's URL.  Link first to the obsidian literature note
#     when one exists, then try to link to a zotero item.  If neither is available don't change the link.

#     Arguments 
#     perplexity_doc: a full pathlib path to a file of markdown coming from perplexity's copy function
#     zot_db_items: a dataframe with a row of info for every zotero DB item.  The columns are: 
#         citekey: the obsidian note citekey (the stem of its filename)
#         zotkey: zoter item key
#         hasLitNote: true if an obsidian literature note already exists
#         url: the URL associated with this zotero DB item
#     output_file: a full pathlib path to where the output document should go"""

#     # organize the zotero DB info
#     if not isinstance(zot_db_items, pd.DataFrame):
#         raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
#         df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
#         zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

#     zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
#     zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

#     # modify the perplexity dialog doc
#     with open(perplexity_doc, 'r') as mdfile:
#         content = mdfile.read()

#     # Split the content into body and citations
#     parts = content.split("\nCitations:\n")
#     if len(parts) != 2:
#         raise Exception("Couldn't find Citations section")
    
#     body, citations = parts

#     # From citations at doc bottom, get a url for each citation number
#     citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
#     doc_number_to_url = defaultdict(lambda: None, {num:normalize_url(url) for num, url in citation_urls})

#     def make_best_reference_link(doc_url, doc_cite_num):
#         # Replace body citations w/ wikilinks to an obsidian note or if no note, an md link to a zotero item
#         if doc_url and (itemInfo := zot_url_to_item_info[doc_url]) is not None:
#             if itemInfo.hasLitNote:
#                 return f'[[{itemInfo.citekey}]]' # wikilink to obsidian lit note

#             # md link to item in zotero DB
#             link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'  #"bob \u2794 jim"
#             return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
            
#         return f'[{doc_cite_num}]' # not in zotero DB so leave unchanged

#     def replace_body_reference(match):
#         # Replace citations in the body text
#         doc_cite_num = match.group(1)
#         doc_url = doc_number_to_url[doc_cite_num]

#         return ' ' + make_best_reference_link(doc_url, doc_cite_num)

#     body = re.sub(r'\[(\d+)\]', replace_body_reference, body)

#     def replace_citations_reference(match):
#         # Replace citations in the Citations section
#         doc_cite_num = match.group(1)
#         url = match.group(2)
#         doc_url = normalize_url(url)

#         if zot_url_to_item_info[doc_url] is None:
#             return f'[{doc_cite_num}] {doc_url}' # not in zotero DB
#         else:
#             return f'[{doc_cite_num}] =={make_best_reference_link(doc_url, doc_cite_num)}== {url}'

#     citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citations_reference, citations)

#     with open(output_file, 'w') as outfile:
#         outfile.write(body + "\nCitations:\n" + citations)
